# Library and Data Importation

In [1]:
#Data manipulation and analysis
import pandas as pd 
import numpy as np

#Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

print("Import Successful!")

Import Successful!


In [2]:
#Importing and loading the 6 datasets 

fact_transactions = pd.read_csv("fact_transactions.csv")
dim_merchant = pd.read_csv("dim_merchant.csv")
dim_account = pd.read_csv("dim_account.csv")
dim_customer = pd.read_csv("dim_customer.csv")
dim_date = pd.read_csv("dim_date.csv")
dim_location = pd.read_csv("dim_location.csv")

print("All files loaded Successfully!")

All files loaded Successfully!


# 1 Transaction Overview & Baseline KPI's

### Overall Transaction overview

In [3]:
#Transactional Overview
print("Total transactions:", len(fact_transactions))
print("Total money moved: $", round(fact_transactions["amount_usd"].sum(), 2))
print("Transaction dates between:", fact_transactions['transaction_date'].min(), "to", fact_transactions["transaction_date"].max())
print("Flagged transaction:", fact_transactions['is_flagged'].sum())


Total transactions: 195276
Total money moved: $ 355057585.98
Transaction dates between: 2024-01-01 to 2024-09-30
Flagged transaction: 19741


### Breakdown by channel

In [4]:
# Grouping transactions by Channel

channel_summary = fact_transactions.groupby('channel').agg(
    total_transaction = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum'),
    flagged = ('is_flagged', 'sum')
).reset_index()

print(channel_summary)

          channel  total_transaction  total_amount  flagged
0             ATM              27104  3.038693e+07      681
1          Branch              13127  2.410850e+07      715
2  Mobile Banking              77973  1.459301e+08    11198
3             POS              21740  2.941751e+07      716
4     Web Banking              55332  1.252146e+08     6431


### Breakdown by Transaction type 

In [5]:
# grouping transaction by type

type_summary = fact_transactions.groupby('transaction_type').agg(
    total_transaction = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum'),
    flagged = ('is_flagged', 'sum')
).reset_index()

print(type_summary)

  transaction_type  total_transaction  total_amount  flagged
0           Credit              29188  1.454919e+07        0
1            Debit              29640  1.469089e+07        0
2      POS Payment              28947  1.467685e+07        0
3         Transfer              38797  5.241851e+07     9523
4    Wire Transfer              39682  2.441420e+08    10218
5       Withdrawal              29022  1.458013e+07        0


### Monthly Trend

In [6]:
# Grouping transaction data by month date

fact_transactions['month'] = pd.to_datetime(fact_transactions['transaction_date']).dt.to_period('M')

monthly = fact_transactions.groupby('month').agg(
    total_transaction = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum'),
    flagged = ('is_flagged', 'sum')
).reset_index()

print(monthly)

     month  total_transaction  total_amount  flagged
0  2024-01              20508   20196692.18      409
1  2024-02              18898   19978083.99      416
2  2024-03              20240   19961346.96      388
3  2024-04              19705   20188180.98      448
4  2024-05              21892   46278110.88     2177
5  2024-06              22049   53041528.43     2728
6  2024-07              24521   60182875.20     4733
7  2024-08              24026   58694512.67     4335
8  2024-09              23437   56536254.69     4107


# 2 Anomaly detection & velocity checks 

### Off-hours activity

In [7]:
# Transactions between 1am and 4am

off_hours = fact_transactions[fact_transactions['is_off_hours'] == 1]

print("Total off-hours transaction:", len(off_hours))
print(f"Transaction amount during off-hours: ${round(off_hours['amount_usd'].sum(), 2)}")
print("Flagged off-hour transactions:", off_hours['is_flagged'].sum())


Total off-hours transaction: 16230
Transaction amount during off-hours: $79476298.17
Flagged off-hour transactions: 10062


### Amount outliers

In [8]:
# Any transaction 3x above the average is suspicious

mean_amount = fact_transactions['amount_usd'].mean()
std_amount = fact_transactions['amount_usd'].std()
threshold = mean_amount + (3 * std_amount)

#defining outliers
outliers = fact_transactions[fact_transactions['amount_usd'] > threshold]


print(f"Normal transaction avg: ${round(mean_amount, 2)}")
print(f"Outlier threshold: ${round(threshold, 2)}")
print(f"Outlier transactions: {len(outliers)}")
print(f"Total outlier amount: ${round(outliers['amount_usd'].sum(), 2)}")

Normal transaction avg: $1818.23
Outlier threshold: $19437.96
Outlier transactions: 5353
Total outlier amount: $177158691.44


### Velocity checks (rapid repeat transactions)

In [9]:
# sort by customer and time

fact_transactions['transaction_datetime'] = pd.to_datetime(fact_transactions['transaction_datetime'])
df_sorted = fact_transactions.sort_values(['customer_key', 'transaction_datetime'])

# Time difference between each transaction per customer (in minutes)
df_sorted['minutes_since_last_txn'] = df_sorted.groupby('customer_key')['transaction_datetime'].diff().dt.total_seconds() / 60

# Flag transactions within 10mins of each other
velocity_flags = df_sorted[df_sorted['minutes_since_last_txn'] <= 10]

print('Rapid Repeat Transactions(within 10 mins):', len(velocity_flags))
print("Unique customers involved:", velocity_flags['customer_key'].nunique())
print(f"Total amount in rapid transactions: ${round(velocity_flags['amount_usd'].sum(), 2)}")

Rapid Repeat Transactions(within 10 mins): 4617
Unique customers involved: 283
Total amount in rapid transactions: $13847767.84


### Geographic mismatch

In [10]:
# Join transactions with customer home and locations

geo_check = fact_transactions.merge(dim_customer[['customer_key', 'location_key']], on = 'customer_key', suffixes =('_txn', '_home'))

# Flag where transaction location differs from customer's home location 
geo_mismatch = geo_check[geo_check['location_key_txn'] != geo_check['location_key_home']]

print("Geographic Mismatch Transactions: ", len(geo_mismatch))
print("Unique customers: ", geo_mismatch['customer_key'].nunique())
print(f"Total amount: ${round(geo_mismatch['amount_usd'].sum(), 2)}")

Geographic Mismatch Transactions:  191165
Unique customers:  7947
Total amount: $351045585.15


# 3 Customer Risk Profiling

### Customer Baseline

In [11]:
# What does each customer normally transact like

customer_baseline = fact_transactions.groupby('customer_key').agg(
    total_transaction = ('transaction_id', 'count'),
    avg_transaction_amount = ('amount_usd', 'mean'),
    max_transaction_amount = ('amount_usd', 'max'),
    total_amount_spent = ('amount_usd', 'sum'),
    total_flagged = ('is_flagged', 'sum'),
    off_hours_spent = ('is_off_hours', 'sum'),
).reset_index()

print(customer_baseline.head(10))

   customer_key  total_transaction  avg_transaction_amount  \
0             2                175            15961.541429   
1             3                 50             2575.056000   
2             4                165            15657.498242   
3             5                 60             2790.121833   
4             6                 74             5308.065676   
5             8                 68             5590.174853   
6             9                 67            14914.048955   
7            10                 83            14881.088554   
8            14                 58            17714.026034   
9            15                 44            13553.227955   

   max_transaction_amount  total_amount_spent  total_flagged  off_hours_spent  
0                49901.97          2793269.75            168               99  
1                 4910.58           128752.80             50               37  
2                49221.23          2583487.21            151               74

### Flagging customers who have outliers from their own history

In [12]:
# Finding anyone who has 3x their own average 
customer_baseline['spike_flag'] = (
    customer_baseline['max_transaction_amount'] > customer_baseline['avg_transaction_amount'] * 3
).astype(int)

# Anyone with more than 3 flagged transactions
customer_baseline['repeat_fraud_flag'] = (
    customer_baseline['total_flagged'] > 3
).astype(int)


# Anyone active off_hours more than 5 times
customer_baseline['off_hours_flag'] = (
    customer_baseline['off_hours_spent'] > 5
).astype(int)

print("Spike flagged customers: ", customer_baseline['spike_flag'].sum())
print("Repeat fraud flag:", customer_baseline['repeat_fraud_flag'].sum())
print("Off-hours flagged:", customer_baseline['off_hours_flag'].sum())

Spike flagged customers:  7216
Repeat fraud flag: 247
Off-hours flagged: 229


### Impossible travel detection (Suspicious)

In [13]:
# Sorting transaction by customer and time 
df_sorted = fact_transactions.sort_values(['customer_key', 'transaction_datetime'])

# Getting each transaction's location vs the previous one for same customer
df_sorted['prev_location'] = df_sorted.groupby('customer_key')['location_key'].shift(1)
df_sorted['minutes_since_last_txn'] = df_sorted.groupby('customer_key')['transaction_datetime'].diff().dt.total_seconds() / 60

# Different locations + within 30 minutes = impossible travel
impossible_travel = df_sorted[
    (df_sorted['location_key'] != df_sorted['prev_location']) & 
    (df_sorted['minutes_since_last_txn'] <= 30)
]

print("Impossible travel events: ", len(impossible_travel))
print("Unique customers: ", impossible_travel['customer_key'].nunique())

Impossible travel events:  5295
Unique customers:  596


### Flagging + Impossible impossible combination per customer 

In [14]:
# Add impossible travel flag to baseline
travel_flags = impossible_travel.groupby('customer_key').size().reset_index(name = 'impossible_travel_count')

customer_profile = customer_baseline.merge(travel_flags, on = 'customer_key', how = 'left')
customer_profile['impossible_travel_count'] = customer_profile['impossible_travel_count'].fillna(0)

customer_profile['impossible_travel_flag'] = (
    customer_profile['impossible_travel_count'] > 0
).astype(int)

# Total risk flags per customer
customer_profile['total_risk_flags'] = (
    customer_profile['spike_flag'] + 
    customer_profile['repeat_fraud_flag'] + 
    customer_profile['off_hours_flag'] +
    customer_profile['impossible_travel_flag']
)

# sort by most suspicious
top_suspects = customer_profile.sort_values('total_risk_flags', ascending = False).head(20)
print(top_suspects[['customer_key', 'total_flagged', 'spike_flag', 'off_hours_flag', 'impossible_travel_flag', 'total_risk_flags']])

     customer_key  total_flagged  spike_flag  off_hours_flag  \
0               2            168           1               1   
73             91            115           1               1   
173           210             98           1               1   
69             85            123           1               1   
68             84            163           1               1   
67             83            149           1               1   
66             81             82           1               1   
64             79            143           1               1   
124           152            117           1               1   
60             75             78           1               1   
177           215             92           1               1   
179           218             77           1               1   
190           233            104           1               1   
191           234            137           1               1   
197           240            112        

# 4 Merchant and Channel Risk Scoring

### Linking merchants to the transactions

In [15]:
txn_merchant = fact_transactions.merge(dim_merchant, on = 'merchant_key', how = 'left')

print(txn_merchant[['transaction_id', 'merchant_name', 'merchant_category', 'is_shell_merchant', 'risk_rating', 'is_flagged']].head(10))

  transaction_id       merchant_name merchant_category  is_shell_merchant  \
0   TXN000100688           FreshPick            Retail                  0   
1   TXN000033632           UrbanEats     Food & Dining                  0   
2   TXN000067075           QuickBite     Food & Dining                  0   
3   TXN000069051          ValueStore            Retail                  0   
4   TXN000075110            TechZone       Electronics                  0   
5   TXN000157053            SmartBuy       Electronics                  0   
6   TXN000061616  International Wire     Wire Transfer                  0   
7   TXN000001060       Domestic Wire     Wire Transfer                  0   
8   TXN000091728           HealthHub        Healthcare                  0   
9   TXN000184120         SpiceGarden     Food & Dining                  0   

  risk_rating  is_flagged  
0         Low           0  
1         Low           0  
2         Low           0  
3         Low           0  
4         Lo

### Scoring Merchants risk factor

In [16]:
#Quick summary of merchants transactions

merchant_risk = txn_merchant.groupby(['merchant_key', 'merchant_name', 'merchant_category', 'is_shell_merchant',
                                     'risk_rating']).agg(
total_transactions = ('transaction_id', 'count'),
total_amount = ('amount_usd', 'sum'),
flagged_transactions = ('is_flagged', 'sum'),
off_hours_transactions = ('is_off_hours', 'sum')
).reset_index()

# Fraud rate 
merchant_risk['fraud_rate'] = (
    merchant_risk['flagged_transactions'] / 
    merchant_risk['total_transactions'] * 100
).round(2)

#Sorting by fraud rate
merchant_risk_sorted = merchant_risk.sort_values('fraud_rate', ascending = False)
print(merchant_risk_sorted.head(10))

    merchant_key       merchant_name merchant_category  is_shell_merchant  \
30            31     GlobalTrade Ltd    Shell Merchant                  1   
37            38  FastBridge Finance    Shell Merchant                  1   
32            33   NovaPay Solutions    Shell Merchant                  1   
33            34       PrimeFin Corp    Shell Merchant                  1   
34            35      SwiftFunds Inc    Shell Merchant                  1   
35            36      TrustEx Global    Shell Merchant                  1   
31            32      Apex Transfers    Shell Merchant                  1   
36            37     ClearPath Remit    Shell Merchant                  1   
41            42       Domestic Wire     Wire Transfer                  0   
10            11        AirTravel Co            Travel                  0   

   risk_rating  total_transactions  total_amount  flagged_transactions  \
30        High                1698   26898463.65                  1698   
37  

### Flagging the dangerous merchants 

In [17]:
#High fraud rate = above 10%

merchant_risk_sorted['is_high_fraud_rate'] = (
    merchant_risk_sorted['fraud_rate'] > 10
).astype(int)

#A merchant is high risk if: A shell merchant OR has a high fraud rate OR is already risk rated high
merchant_risk_sorted['merchant_risk_flag'] = (
    (merchant_risk_sorted['is_shell_merchant'] == 1) |
    (merchant_risk_sorted['is_high_fraud_rate'] == 1) |
    (merchant_risk_sorted['risk_rating'] == 'High')
).astype(int)

risky_merchants = merchant_risk_sorted[merchant_risk_sorted['merchant_risk_flag'] == 1]
print(f"High risk merchants: {len(risky_merchants)}")
print(risky_merchants[['merchant_name', 'merchant_category', 'fraud_rate',
                      'is_shell_merchant', 'risk_rating']].head(10))

High risk merchants: 8
         merchant_name merchant_category  fraud_rate  is_shell_merchant  \
30     GlobalTrade Ltd    Shell Merchant       100.0                  1   
37  FastBridge Finance    Shell Merchant       100.0                  1   
32   NovaPay Solutions    Shell Merchant       100.0                  1   
33       PrimeFin Corp    Shell Merchant       100.0                  1   
34      SwiftFunds Inc    Shell Merchant       100.0                  1   
35      TrustEx Global    Shell Merchant       100.0                  1   
31      Apex Transfers    Shell Merchant       100.0                  1   
36     ClearPath Remit    Shell Merchant       100.0                  1   

   risk_rating  
30        High  
37        High  
32        High  
33        High  
34        High  
35        High  
31        High  
36        High  


### Channel risk scoring 

In [18]:
channel_risk = fact_transactions.groupby('channel').agg(
    total_transactions = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum'),
    flagged_transaction = ('is_flagged', 'sum'),
    off_hours_transaction = ('is_off_hours', 'sum')
).reset_index()

#Fraud rate per channel
channel_risk['fraud_rate'] = (
    channel_risk['flagged_transaction'] / 
    channel_risk['total_transactions'] * 100
).round(2)

#Off hours rate per channel
channel_risk['off_hours_rate'] = (
    channel_risk['off_hours_transaction'] / 
    channel_risk['total_transactions'] * 100
).round(2)

channel_risk_sorted = channel_risk.sort_values('fraud_rate', ascending = False)
print(channel_risk_sorted)

          channel  total_transactions  total_amount  flagged_transaction  \
2  Mobile Banking               77973  1.459301e+08                11198   
4     Web Banking               55332  1.252146e+08                 6431   
1          Branch               13127  2.410850e+07                  715   
3             POS               21740  2.941751e+07                  716   
0             ATM               27104  3.038693e+07                  681   

   off_hours_transaction  fraud_rate  off_hours_rate  
2                   8956       14.36           11.49  
4                   4817       11.62            8.71  
1                    540        5.45            4.11  
3                    880        3.29            4.05  
0                   1037        2.51            3.83  


### Which merchant categories are riskiest ?

In [19]:
category_risk = txn_merchant.groupby('merchant_category').agg(
    total_transactions = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum'),
    off_hours_transactions = ('is_off_hours', 'sum'),
    flagged_transactions = ('is_flagged', 'sum')
).reset_index()

category_risk['fraud_rate'] = (
    category_risk['flagged_transactions'] /
    category_risk['total_transactions'] * 100
).round(2)

category_risk_sorted = category_risk.sort_values('fraud_rate', ascending = False)
print(category_risk_sorted)

  merchant_category  total_transactions  total_amount  off_hours_transactions  \
7    Shell Merchant               13671  2.162971e+08                    5940   
9     Wire Transfer                9742  7.499524e+06                     548   
6            Retail               24588  1.920078e+07                    1397   
8            Travel               24388  1.849245e+07                    1400   
0               ATM               14731  1.112289e+07                     859   
3     Food & Dining               24650  1.871841e+07                    1409   
2     Entertainment               24514  1.886942e+07                    1388   
1       Electronics               24635  1.884704e+07                    1410   
4        Healthcare               24622  1.880773e+07                    1298   
5               POS                9735  7.202277e+06                     581   

   flagged_transactions  fraud_rate  
7                 13671      100.00  
9                   349        3

In [20]:
txn_merchant = txn_merchant.merge(
    merchant_risk_sorted[['merchant_key', 'merchant_risk_flag', 'fraud_rate']], on = 'merchant_key', how = 'left'
)


txn_merchant['merchant_risk_flag'] = txn_merchant['merchant_risk_flag'].fillna(0)

print('Done, Columns now', txn_merchant.columns.tolist())

Done, Columns now ['transaction_key', 'transaction_id', 'customer_key', 'account_key', 'merchant_key', 'location_key', 'date_key', 'transaction_datetime', 'transaction_date', 'transaction_hour', 'transaction_type', 'channel', 'amount_usd', 'currency', 'is_flagged', 'fraud_type', 'is_off_hours', 'status', 'month', 'merchant_name', 'merchant_category', 'is_shell_merchant', 'risk_rating', 'country', 'merchant_risk_flag', 'fraud_rate']


# 5 Fraud risk scoring

### Merging customers profile and risky merchants 

In [21]:
# Get customers who transacted with risky merchants 

risky_merchant_customers = txn_merchant[
    txn_merchant['merchant_risk_flag'] == 1
].groupby('customer_key').agg(
    risky_merchant_txn_count = ('transaction_id', 'count'),
    risky_merchant_amount = ('amount_usd', 'sum')
).reset_index()

#Merge into customer profile

customer_scored = customer_profile.merge(
    risky_merchant_customers, on = 'customer_key', how = 'left'
)


customer_scored['risky_merchant_txn_count'] = customer_scored['risky_merchant_txn_count'].fillna(0)
customer_scored['risky_merchant_amount'] = customer_scored['risky_merchant_amount'].fillna(0)

#Flag customers with risky merchants transactions
customer_scored['risky_merchant_flag'] = (customer_scored['risky_merchant_txn_count'] > 0).astype(int)

print(customer_scored.head(5))

   customer_key  total_transaction  avg_transaction_amount  \
0             2                175            15961.541429   
1             3                 50             2575.056000   
2             4                165            15657.498242   
3             5                 60             2790.121833   
4             6                 74             5308.065676   

   max_transaction_amount  total_amount_spent  total_flagged  off_hours_spent  \
0                49901.97          2793269.75            168               99   
1                 4910.58           128752.80             50               37   
2                49221.23          2583487.21            151               74   
3                 4995.32           167407.31             60               39   
4                 9710.48           392796.86             74               74   

   spike_flag  repeat_fraud_flag  off_hours_flag  impossible_travel_count  \
0           1                  1               1               

### Building a weighted risk score 

In [22]:
#Weighted score per customer 
customer_scored['risk_score'] = (
    (customer_scored['repeat_fraud_flag'] * 40) + 
    (customer_scored['impossible_travel_flag'] * 30) +
    (customer_scored['spike_flag'] * 20) +
    (customer_scored['off_hours_flag'] * 10)+
    (customer_scored['risky_merchant_flag'] * 15)
)

print('Risk score distribution:')
print(customer_scored['risk_score'].value_counts().sort_index(ascending = False).head(10))

Risk score distribution:
risk_score
115      50
95      142
85        3
65       36
55       16
50      381
30       21
20     6784
0       514
Name: count, dtype: int64


### Assigning risk tiers using NTILE logic

In [23]:
customer_scored['risk_tier'] = pd.cut(
    customer_scored['risk_score'],
    bins = [-1, 0, 20, 60, 115],
    labels = ['Low', 'Medium', 'High', 'Critical']
)

print(customer_scored['risk_tier'].value_counts())

risk_tier
Medium      6784
Low          514
High         418
Critical     231
Name: count, dtype: int64


### Add customers name to the watchlist

In [24]:
# Bringing in customers names and details 

watchlist = customer_scored.merge(
    dim_customer[['customer_key', 'customer_id', 'full_name', 'country', 
                 'kyc_status', 'customer_segment']], 
    on = 'customer_key', how = 'left'
)

# Final watchlist - Critical and High risk only
final_watchlist = watchlist[
    watchlist['risk_tier'].isin(['Critical', 'High'])
].sort_values('risk_score', ascending = False)

print(f"Critical risk customers: {len(final_watchlist[final_watchlist['risk_tier'] == 'Critical'])}")
print(f"High risk customers: {len(final_watchlist[final_watchlist['risk_tier'] == 'High'])}\n")
print(final_watchlist[['customer_id', 'full_name', 'country', 'risk_score', 'risk_tier', 'total_flagged',
                      'kyc_status']].head(20))

Critical risk customers: 231
High risk customers: 418

     customer_id         full_name         country  risk_score risk_tier  \
0    NAB-C000002        Ronke Zulu         Nigeria         115  Critical   
90   NAB-C000113      Faith Ikenna         Nigeria         115  Critical   
114  NAB-C000141        Dele Lewis         Nigeria         115  Critical   
108  NAB-C000134  Ibrahim Abubakar         Nigeria         115  Critical   
106  NAB-C000132      Lindiwe Wike         Nigeria         115  Critical   
104  NAB-C000130      Aisha Hughes         Nigeria         115  Critical   
103  NAB-C000129       Garba Saleh          France         115  Critical   
101  NAB-C000126      Ngozi Okafor         Nigeria         115  Critical   
99   NAB-C000123    Amelia Nxumalo         Nigeria         115  Critical   
96   NAB-C000120     David Nxumalo           Kenya         115  Critical   
89   NAB-C000112         Sade Zulu         Nigeria         115  Critical   
127  NAB-C000155        Isla Lewi

### Saving final watchlist

In [25]:
final_watchlist.to_csv("final_watchlist.csv", index = False)
print("watchlist saved as fraud_watchlist.csv")

watchlist saved as fraud_watchlist.csv


# 6 Executive risk report and recommendation

### Exposure from Critical and High fraud tier 

In [26]:
#Amount of money tied to the risky accounts

exposure = final_watchlist.groupby('risk_tier').agg(
    customer_count = ('customer_id', 'count'),
    total_flagged_transaction = ('total_flagged', 'sum'),
    total_amount_at_risk = ('total_amount_spent', 'sum')
).reset_index()

print(exposure)

  risk_tier  customer_count  total_flagged_transaction  total_amount_at_risk
0       Low               0                          0          0.000000e+00
1    Medium               0                          0          0.000000e+00
2      High             418                        530          1.404461e+07
3  Critical             231                      19211          2.582976e+08


### Top Accounts to Freeze

In [29]:
#Top accounts to freeze

freeze_list = final_watchlist[
    final_watchlist['risk_tier'] == 'Critical'
].sort_values(['risk_score', 'total_flagged'], ascending = [False, False])[
    ['customer_id', 'full_name', 'country', 'risk_score',
    'total_flagged', 'total_amount_spent', 'kyc_status']
].head(20)

print('TOP 20 ACCOUNTS TO FREEZE IMMEDIATELY')
print(freeze_list.to_string())

TOP 20 ACCOUNTS TO FREEZE IMMEDIATELY
     customer_id         full_name         country  risk_score  total_flagged  total_amount_spent kyc_status
227  NAB-C000279         Kofi Sani         Nigeria         115            169          2640439.47   Verified
0    NAB-C000002        Ronke Zulu         Nigeria         115            168          2793269.75   Verified
104  NAB-C000130      Aisha Hughes         Nigeria         115            165          2752171.16    Expired
68   NAB-C000084     Samuel Taylor         Nigeria         115            163          2776207.71   Verified
114  NAB-C000141        Dele Lewis         Nigeria         115            159          1720281.89   Verified
2    NAB-C000004      Aisha Wilson   United States         115            151          2583487.21   Verified
67   NAB-C000083     Joseph Yakubu  United Kingdom         115            149          2223115.62   Verified
64   NAB-C000079       Ahmed Davis           Kenya         115            143          210

### Channel restriction recommendation

In [41]:
#Suggested channels to be limited and pay attention to

channel_restriction = fact_transactions[
    fact_transactions['customer_key'].isin(
    final_watchlist['customer_key']
    )
].groupby('channel').agg(
    flagged_transaction = ('is_flagged', 'sum'),
    total_transaction = ('transaction_id', 'count'),
    total_amount = ('amount_usd', 'sum')
).reset_index()

channel_restriction['fraud_rate'] = (
    channel_restriction['flagged_transaction'] / channel_restriction['total_transaction'] * 100
).round(2)

print(channel_restriction.sort_values('fraud_rate', ascending = False))

          channel  flagged_transaction  total_transaction  total_amount  \
2  Mobile Banking                11198              15387  1.146270e+08   
4     Web Banking                 6431               9282  1.023396e+08   
1          Branch                  715               1404  1.821281e+07   
3             POS                  716               1896  1.933948e+07   
0             ATM                  681               2158  1.782331e+07   

   fraud_rate  
2       72.78  
4       69.28  
1       50.93  
3       37.76  
0       31.56  


# Full Report Summary

In [72]:
total_customers = len(dim_customer)
critical_count = len(final_watchlist[final_watchlist['risk_tier'] == 'Critical'])
high_count = len(final_watchlist[final_watchlist['risk_tier'] == 'High'])
total_flagged_txn = final_watchlist['total_flagged'].sum()
total_exposure = exposure['total_amount_at_risk'].sum()


def divider(width=60):
    return "\u2500" * width

w = 60

report = f"""

{divider(w)}
OPERATION CLEAR WATER - EXECUTIVE RISK REPORT
NorthAxis Bank | Risk Intelligence Division
Period: Jan - Sep 2024 | REF: NAB-RI-2024-09
{divider(w)}


EXECUTIVE SUMMARY
{divider(w)}

Total Customers Analysed  : {total_customers:,}
Critical Risk Accounts   : {critical_count:,} ← TO BE FROZEN IMMEDIATELY
High Risk Accounts    : {high_count:,} ← TO BE MONITORED CLOSELY
Total Flagged Transactions : {total_flagged_txn:,}
Estimated Amount at Risk  : ${total_exposure:,.2f}


KEY FRAUD SIGNALS DETECTED
{divider(w)}

1. Velocity Anomalies: Multiple transaction within minutes
2. Off-hours Activity: Suspicious 1AM-4AM transaction spikes
3. Impossible Travel: Same customer different location in less than 30 minutes
4. Amount Spikes: Suddenly doing a transaction more than 3 times their own transaction average
5. Shell Merchant Links: Transactions rooted through flagged merchants


RECOMMENDATION
{divider(w)}

1. FREEZE - Immediately Suspend {critical_count} Critical accounts 
2. RESTRICT - Limit digital channel transactions above $500 pending review
3. REVIEW - KYC re-verification for all High & Critical tier customers
4. MONITOR - Real-time velocity alerts for accounts scoring above 20
5. ESCALATE - Refer critical accounts to regulatory compliance team.

{divider(w)}
PREPARED BY : Risk Intelligence Division
CONFIDENTIAL - FOR BOARD USE ONLY

"""

print(report)



────────────────────────────────────────────────────────────
OPERATION CLEAR WATER - EXECUTIVE RISK REPORT
NorthAxis Bank | Risk Intelligence Division
Period: Jan - Sep 2024 | REF: NAB-RI-2024-09
────────────────────────────────────────────────────────────


EXECUTIVE SUMMARY
────────────────────────────────────────────────────────────

Total Customers Analysed  : 8,000
Critical Risk Accounts   : 231 ← TO BE FROZEN IMMEDIATELY
High Risk Accounts    : 418 ← TO BE MONITORED CLOSELY
Total Flagged Transactions : 19,741
Estimated Amount at Risk  : $272,342,236.90


KEY FRAUD SIGNALS DETECTED
────────────────────────────────────────────────────────────

1. Velocity Anomalies: Multiple transaction within minutes
2. Off-hours Activity: Suspicious 1AM-4AM transaction spikes
3. Impossible Travel: Same customer different location in less than 30 minutes
4. Amount Spikes: Suddenly doing a transaction more than 3 times their own transaction average
5. Shell Merchant Links: Transactions rooted thr

# Report Files

In [75]:
# Exporting report files

freeze_list.to_csv("freeze_list.csv", index = False)
channel_restriction.to_csv("channel_restriction.csv", index = False)
exposure.to_csv("exposure_details.csv")

# Saving the report as a text file

with open('executive_risk_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)
    
print("All Files Saved successfully!")
print("Files produced:")
print("- fraud_watchlist.csv")
print("- freeze_list_top20.csv")
print("- channel_restriction_summary.csv")
print("- exposure_by_tier.csv")

All Files Saved successfully
Files produced:
- fraud_watchlist.csv
- freeze_list_top20.csv
- channel_restriction_summary.csv
- exposure_by_tier.csv
